# SONIC → GR00T-Lite training preflight (Colab)

This notebook mounts Drive, safely updates the current repository, validates the exact G1 MJCF and BONES timing/joint/FK semantics, renders reference videos, benchmarks MJWarp, smoke-tests SONIC, evaluates held-out tracking, then freezes SONIC before encoding captioned tokens and training GR00T-Lite. It deliberately does **not** jointly train SONIC and GR00T.

Use a GPU runtime. Full BONES processing and SONIC training are long-running jobs; outputs and checkpoints are copied to Drive. Do not enable the full-run toggles until all preceding gates pass.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Edit only these values. The MJCF must be the exact model used for training/deployment.
REPO_URL = "https://github.com/YigitGunduc/robot.git"
BRANCH = "master"
REPO = "/content/robot"
BONES_DRIVE = "/content/drive/MyDrive/Datasets/bones-seed"
MJCF_DRIVE = (
    "/content/drive/MyDrive/sonic_groot_assets/nvidia_g1/mjcf/g1_29dof_rev_1_0.xml"
)
PERSIST = "/content/drive/MyDrive/sonic_groot_artifacts"
WORK = "/content/sonic_groot_work"
# First-run default. Change to False only after all packed Drive caches exist.
RUN_DATA_PREPROCESS = True
RUN_SONIC_SMOKE = False
RUN_FULL_SONIC = False
RUN_GROOT_SMOKE = False
RUN_FULL_GROOT = False

In [ ]:
# Safe clone/update: preserve local edits and permit only a fast-forward pull.
import shlex
import subprocess
from collections import deque
from pathlib import Path


def run_checked(command, *, cwd=None, env=None, capture_output=False):
    """Stream command output and include its diagnostic tail in any error."""
    command = [str(part) for part in command]
    printable = shlex.join(command)
    print(f"$ {printable}", flush=True)
    tail = deque(maxlen=200)
    captured = [] if capture_output else None
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError(f"Unable to capture output for: {printable}")
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if captured is not None:
            captured.append(line)
    return_code = process.wait()
    if return_code != 0:
        diagnostic = "".join(tail).rstrip() or "<command produced no output>"
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {printable}\n"
            f"Last {len(tail)} output lines (full output is above):\n{diagnostic}"
        )
    return "".join(captured) if captured is not None else return_code


repo = Path(REPO)
if (repo / ".git").is_dir():
    dirty = run_checked(
        ["git", "-C", REPO, "status", "--porcelain"], capture_output=True
    ).strip()
    if dirty:
        raise RuntimeError(
            "Colab repository has local changes; refusing git pull:\n" + dirty
        )
    run_checked(["git", "-C", REPO, "pull", "--ff-only", "origin", BRANCH])
else:
    run_checked(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO],
    )
run_checked(["git", "-C", REPO, "status", "--short", "--branch"])

In [ ]:
run_checked(["apt-get", "update", "-qq"])
run_checked(["apt-get", "install", "-y", "-qq", "zstd", "ffmpeg", "rsync"])
run_checked(["python", "-m", "pip", "install", "-q", "-U", "pip"])
run_checked(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        REPO + "[sim,hf,dev,video]",
    ],
)

In [ ]:
# Repository and runtime quality gates.
import os
import shutil

import torch

os.chdir(REPO)
run_checked(
    ["python", "-m", "compileall", "-q", "gear_sonic_mjx", "groot_lite", "scripts"],
)
run_checked(["python", "-m", "ruff", "check", "."])
run_checked(["python", "-m", "pytest", "-q", "-ra", "--tb=short"])
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))
Path(PERSIST).mkdir(parents=True, exist_ok=True)
Path(WORK).mkdir(parents=True, exist_ok=True)
if not Path(MJCF_DRIVE).is_file():
    raise FileNotFoundError(
        f"Missing exact 29-DOF G1 MJCF: {MJCF_DRIVE}. "
        "Keep the sibling meshes/g1 directory when copying this asset."
    )

## Data staging
The known Drive dataset path is `MyDrive/Datasets/bones-seed`, with `g1.tar.zst`, `seed_metadata_v004.parquet`, and `seed_metadata_v002_temporal_labels.jsonl`. The archive is copied/decompressed to Colab's local SSD because training directly from thousands of Drive files is too slow. The disk-space check is intentional.

In [ ]:
BONES_ARCHIVE = str(Path(BONES_DRIVE) / "g1.tar.zst")
METADATA = str(Path(BONES_DRIVE) / "metadata/seed_metadata_v004.parquet")
TIMELINES = str(Path(BONES_DRIVE) / "metadata/seed_metadata_v002_temporal_labels.jsonl")
for required in [BONES_ARCHIVE, METADATA, TIMELINES]:
    path = Path(required)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing/empty Drive input: {path}")
archive_gb = Path(BONES_ARCHIVE).stat().st_size / 1e9
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"archive={archive_gb:.1f} GB, local free={free_gb:.1f} GB")
if free_gb < max(70, archive_gb * 3):
    raise RuntimeError(
        "Not enough local SSD to safely extract/process BONES; use a larger Colab runtime/disk"
    )

In [ ]:
RAW = str(Path(WORK) / "bones_raw")
PREP = str(Path(WORK) / "bones_30hz")
SPLITS = str(Path(WORK) / "bones_splits.json")
if RUN_DATA_PREPROCESS is None:
    cached_split_metadata = [
        Path(PERSIST) / f"bones_50hz_{split}" / "_packed_metadata.npz"
        for split in ["train", "validation", "test"]
    ]
    RUN_DATA_PREPROCESS = not all(path.is_file() for path in cached_split_metadata)
    print(
        "Data mode:",
        "preprocess BONES archive"
        if RUN_DATA_PREPROCESS
        else "reuse complete packed Drive caches",
    )
if RUN_DATA_PREPROCESS:
    Path(RAW).mkdir(parents=True, exist_ok=True)
    marker = Path(RAW) / ".extracted"
    if not marker.exists():
        run_checked(
            ["tar", "--use-compress-program=unzstd", "-xf", BONES_ARCHIVE, "-C", RAW],
        )
        marker.touch()
    csv_count = sum(1 for _ in Path(RAW).rglob("*.csv"))
    if csv_count == 0:
        raise RuntimeError("Archive extracted but no BONES CSV files were found")
    print("BONES CSV files:", csv_count)
    if not (Path(PREP) / "_manifest.npz").is_file():
        run_checked(
            [
                "python",
                "scripts/preprocess_bones.py",
                "--input",
                RAW,
                "--output",
                PREP,
            ],
        )
    else:
        print("Reusing completed local 30-Hz cache:", PREP)
    if not Path(SPLITS).is_file():
        run_checked(
            [
                "python",
                "scripts/split_bones.py",
                "--motions",
                PREP,
                "--output",
                SPLITS,
                "--seed",
                "0",
            ],
        )
    else:
        print("Reusing deterministic split manifest:", SPLITS)
else:
    print(
        "Preprocessing was disabled; the next cell requires complete packed caches in PERSIST."
    )

In [ ]:
# Static physical preflight before expensive 50-Hz FK packing.
PREFLIGHT = str(Path(PERSIST) / "preflight_mjcf_report.json")
MOTION_PREFLIGHT = (
    PREP if Path(PREP).is_dir() else str(Path(PERSIST) / "bones_50hz_train")
)
if not Path(MOTION_PREFLIGHT).is_dir():
    raise FileNotFoundError(
        "No BONES motion library is available for preflight.\n"
        f"RUN_DATA_PREPROCESS={RUN_DATA_PREPROCESS!r}\n"
        f"Missing local preprocessed directory: {PREP}\n"
        f"Missing cached packed directory: {Path(PERSIST) / 'bones_50hz_train'}\n"
        "For the first run, set RUN_DATA_PREPROCESS=True (or None) in the configuration "
        "cell, then rerun the data-staging and preprocessing cells."
    )
run_checked(
    [
        "python",
        "scripts/preflight_training.py",
        "--mjcf",
        MJCF_DRIVE,
        "--output",
        PREFLIGHT,
    ],
)

In [ ]:
# Render category-diverse references. Check left/right, root height, feet, limits, and caption meaning.
import numpy as np

from gear_sonic_mjx.envs.motion_library import open_motion_library

library = open_motion_library(MOTION_PREFLIGHT, 50.0)
motion_names = (
    library.names
    if hasattr(library, "names")
    else [path.stem for path in library.files]
)
selected = []
for keyword in [
    "walk",
    "run",
    "turn",
    "crouch",
    "kneel",
    "crawl",
    "jump",
    "wave",
    "dance",
    "stand",
]:
    match = next(
        (index for index, name in enumerate(motion_names) if keyword in name.lower()),
        None,
    )
    if match is not None and match not in selected:
        selected.append(match)
for index in (
    np.linspace(0, len(library) - 1, min(12, len(library))).round().astype(int)
):
    if len(selected) >= 12:
        break
    if int(index) not in selected:
        selected.append(int(index))
video_dir = Path(PERSIST) / "reference_videos"
video_dir.mkdir(parents=True, exist_ok=True)
for motion_id in selected:
    run_checked(
        [
            "python",
            "scripts/render_reference_motion.py",
            "--mjcf",
            MJCF_DRIVE,
            "--motions",
            MOTION_PREFLIGHT,
            "--motion-id",
            str(motion_id),
            "--output",
            str(video_dir / f"motion_{motion_id:06d}.mp4"),
            "--max-frames",
            "500",
        ],
    )
print("Inspect videos in:", video_dir)

In [ ]:
# Pack each leakage-safe split into contiguous memory-mapped arrays.
PACKED = {}
if RUN_DATA_PREPROCESS:
    for split in ["train", "validation", "test"]:
        destination = str(Path(WORK) / f"bones_50hz_{split}")
        pack_command = [
            "python",
            "scripts/pack_bones_mmap.py",
            "--motions",
            PREP,
            "--output",
            destination,
            "--fps",
            "50",
            "--mjcf",
            MJCF_DRIVE,
            "--split-manifest",
            SPLITS,
            "--split",
            split,
        ]
        if split == "train":
            pack_command += ["--nvidia-upper-body-augment", "--upper-body-prob", "0.5"]
        run_checked(pack_command)
        PACKED[split] = destination
        run_checked(
            [
                "rsync",
                "-a",
                destination + "/",
                str(Path(PERSIST) / f"bones_50hz_{split}") + "/",
            ],
        )
    shutil.copy2(SPLITS, Path(PERSIST) / "bones_splits.json")
else:
    for split in ["train", "validation", "test"]:
        PACKED[split] = str(Path(PERSIST) / f"bones_50hz_{split}")
print(PACKED)
# Now validate finite values, velocities, limits, penetration, and FK on each final 50-Hz split.
for split, motions in PACKED.items():
    if not Path(motions).is_dir():
        raise FileNotFoundError(f"Missing packed {split} split: {motions}")
    run_checked(
        [
            "python",
            "scripts/preflight_training.py",
            "--mjcf",
            MJCF_DRIVE,
            "--motions",
            motions,
            "--max-clips",
            "100",
            "--output",
            str(Path(PERSIST) / f"preflight_{split}.json"),
        ],
    )

In [ ]:
# GPU capacity/throughput gate. This also fails on MJWarp contact-buffer overflow.
import json

BENCHMARK = str(Path(PERSIST) / "mjwarp_benchmark.json")
run_checked(
    [
        "python",
        "scripts/benchmark_mjwarp.py",
        "--mjcf",
        MJCF_DRIVE,
        "--worlds",
        "64",
        "256",
        "1024",
        "2048",
        "4096",
        "--output",
        BENCHMARK,
    ],
)
with open(BENCHMARK) as stream:
    WORLD_COUNT = int(json.load(stream)["recommended_worlds_for_physics_throughput"])
print("Benchmark-selected world count:", WORLD_COUNT)

## SONIC first
Run a small no-randomization smoke test, evaluate the validation split, then increase to 256/1024 worlds before the full run. A smoke checkpoint is not expected to meet the tracking gate. The serious run should reach at least 95% held-out success and <40 mm local MPJPE before token export; the intended final target is >97% and about 30 mm where metrics are comparable.

In [ ]:
SMOKE_RUN = str(Path(PERSIST) / "sonic_smoke")
if RUN_SONIC_SMOKE:
    run_checked(
        [
            "python",
            "scripts/train_sonic_mjwarp.py",
            "--mjcf",
            MJCF_DRIVE,
            "--motions",
            PACKED["train"],
            "--network",
            "small",
            "--num-envs",
            "64",
            "--iterations",
            "20",
            "--save-interval",
            "10",
            "--no-domain-randomization",
            "--output",
            SMOKE_RUN,
        ],
    )
    smoke_ckpt = max(Path(SMOKE_RUN).glob("checkpoint_*.pt"))
    run_checked(
        [
            "python",
            "scripts/evaluate_sonic_mjwarp.py",
            "--mjcf",
            MJCF_DRIVE,
            "--motions",
            PACKED["validation"],
            "--checkpoint",
            str(smoke_ckpt),
            "--output",
            str(Path(SMOKE_RUN) / "validation.json"),
            "--num-envs",
            "64",
        ],
    )

In [ ]:
# Start or resume the full SONIC run only after preflight, videos, benchmark, and staged convergence checks pass.
SONIC_RUN = str(Path(PERSIST) / "sonic_small")
if RUN_FULL_SONIC:
    existing = sorted(Path(SONIC_RUN).glob("checkpoint_*.pt"))
    command = [
        "python",
        "scripts/train_sonic_mjwarp.py",
        "--mjcf",
        MJCF_DRIVE,
        "--motions",
        PACKED["train"],
        "--network",
        "small",
        "--num-envs",
        str(WORLD_COUNT),
        "--output",
        SONIC_RUN,
    ]
    if existing:
        command += ["--resume", str(existing[-1])]
    run_checked(command)

In [ ]:
# Hard held-out gate before SONIC is frozen and exposed to GR00T.
if RUN_FULL_SONIC:
    SONIC_CKPT = str(max(Path(SONIC_RUN).glob("checkpoint_*.pt")))
    run_checked(
        [
            "python",
            "scripts/evaluate_sonic_mjwarp.py",
            "--mjcf",
            MJCF_DRIVE,
            "--motions",
            PACKED["validation"],
            "--checkpoint",
            SONIC_CKPT,
            "--output",
            str(Path(SONIC_RUN) / "validation_final.json"),
            "--metadata",
            METADATA,
            "--min-success",
            "0.95",
            "--max-local-mpjpe-mm",
            "40",
            "--fail-on-gate",
        ],
    )
else:
    SONIC_CKPT = "SET_TO_A_GATE_PASSING_SONIC_CHECKPOINT"

## Freeze SONIC, then train GR00T-Lite
Captions enter only now. Each split is encoded independently using the frozen SONIC G1 encoder/FSQ. `--require-captions` makes annotation mismatches a hard failure instead of silently training filename supervision. Export from the clean preprocessed clips—not the upper-body-augmented SONIC training pack—so captions and motion remain aligned. BONES Stage A is text-to-whole-body motion; it does not provide synchronized camera manipulation demonstrations.

In [ ]:
TOKEN_DATA = {}
if RUN_GROOT_SMOKE or RUN_FULL_GROOT:
    if not Path(SONIC_CKPT).is_file():
        raise FileNotFoundError("Set SONIC_CKPT to a held-out-gate-passing checkpoint")
    if not Path(PREP).is_dir():
        raise FileNotFoundError(
            "Clean preprocessed BONES clips are required for caption-aligned token export; rerun the data-preprocess cells"
        )
    split_manifest = (
        SPLITS if Path(SPLITS).is_file() else str(Path(PERSIST) / "bones_splits.json")
    )
    if not Path(split_manifest).is_file():
        raise FileNotFoundError("Missing deterministic split manifest")
    for split in ["train", "validation"]:
        destination = str(Path(PERSIST) / f"bones_sonic_tokens_{split}")
        run_checked(
            [
                "python",
                "scripts/encode_bones_tokens.py",
                "--motions",
                PREP,
                "--checkpoint",
                SONIC_CKPT,
                "--metadata",
                METADATA,
                "--timelines",
                TIMELINES,
                "--split-manifest",
                split_manifest,
                "--split",
                split,
                "--require-captions",
                "--output",
                destination,
            ],
        )
        TOKEN_DATA[split] = destination

In [ ]:
GROOT_RUN = str(Path(PERSIST) / "groot_lite")
if RUN_GROOT_SMOKE:
    run_checked(
        [
            "python",
            "-m",
            "groot_lite.train",
            "--data",
            TOKEN_DATA["train"],
            "--validation-data",
            TOKEN_DATA["validation"],
            "--output",
            GROOT_RUN + "_smoke",
            "--batch-size",
            "8",
            "--max-steps",
            "100",
            "--num-workers",
            "2",
        ],
    )
if RUN_FULL_GROOT:
    existing = sorted(Path(GROOT_RUN).glob("step_*.pt"))
    command = [
        "python",
        "-m",
        "groot_lite.train",
        "--data",
        TOKEN_DATA["train"],
        "--validation-data",
        TOKEN_DATA["validation"],
        "--output",
        GROOT_RUN,
    ]
    if existing:
        command += ["--resume", str(existing[-1])]
    run_checked(command)

## Stop condition
At this point the result is **text + current G1 state → future frozen SONIC tokens → stable 29-joint targets**. Do not claim visual manipulation from BONES. That requires a separate synchronized demonstration dataset containing RGB, instruction, robot state, and successful SONIC/hand actions. Joint fine-tuning is a later experiment; keep the SONIC encoder, FSQ, and token definition frozen.